# First Look at pm-edge Captured Data

Goals: verify data quality on the local archive, get a feel for the universe, look for obvious patterns or oddities at multiple timescales.

In [1]:
# ruff: noqa: F401, I001
import os

os.environ.setdefault(
    "PM_EDGE_LOCAL_FORWARD_INDEX_DIR",
    "/Users/larrymitchell/pm-edge-data/forward_index",
)

from datetime import datetime, timedelta

import pandas as pd

from notebooks.lib.queries import (
    analysis_readiness,
    book_state_quality,
    book_validity,
    capture_freshness,
    data_integrity_checks,
    get_connection,
    hourly_snapshot_volume,
    market_movement,
    metadata_universe,
    multi_timescale_aggregation,
    polymarket_category_expression,
    snapshot_gaps,
    source_breakdown,
    spread_depth_summary,
    trade_overview,
)
from notebooks.lib.plots import (
    plot_book_top_over_time,
    plot_hourly_volume,
    plot_movement_distribution,
    plot_multi_timescale_overlay,
    plot_spread_distribution,
)

con = get_connection()

## Section 1: Capture health overview

In [2]:
df = hourly_snapshot_volume(con)
plot_hourly_volume(df)

In [3]:
source_breakdown(con)

,venue,snapshot_source,count,pct_of_venue
0,kalshi,reconstructed_from_complement,215126,1.000000
1,polymarket,websocket,4752068,0.997844
2,polymarket,rest,10267,0.002156


## Section 2: Data quality verification

In [4]:
book_state_quality(con, venue="polymarket")

,snapshots,avg_bid_levels,avg_ask_levels,pct_with_top_bid,pct_with_top_ask,mean_top_bid,mean_top_ask
0,4762335,3.386008,4.929419,0.810974,0.995724,0.114785,0.097576


In [5]:
book_state_quality(con, venue="kalshi")

,snapshots,avg_bid_levels,avg_ask_levels,pct_with_top_bid,pct_with_top_ask,mean_top_bid,mean_top_ask
0,215126,4.513941,4.505262,0.933458,0.943977,0.49516,0.45597


## Section 3: Integrity, liquidity, and readiness

These checks answer whether the archive is complete enough to analyze. They look for stale capture, missing time buckets, invalid books, shallow liquidity, trade coverage, metadata coverage, duplicate keys, and markets that pass a basic analysis-readiness screen.


In [6]:
capture_freshness(con)


,venue,first_snapshot_utc,latest_snapshot_utc,snapshot_age_minutes,snapshots,distinct_markets,observed_hours,expected_snapshots_at_cadence,pct_expected_snapshots
0,kalshi,2026-05-14 19:47:01.663487-04:00,2026-05-16 21:44:29.795213-04:00,672.6,215126,313,49.957778,3.752828e+06,0.057324
1,polymarket,2026-05-14 19:47:01.663487-04:00,2026-05-16 21:44:29.795213-04:00,672.6,4762335,963,49.957778,1.154624e+07,0.412458


In [7]:
snapshot_gaps(con).head(25)


,venue,bucket,snapshot_count,venue_median,threshold_count
0,kalshi,2026-05-14 22:20:00-04:00,121,285.0,142.5
1,kalshi,2026-05-14 22:25:00-04:00,40,285.0,142.5
2,kalshi,2026-05-14 22:30:00-04:00,40,285.0,142.5
3,kalshi,2026-05-14 22:35:00-04:00,40,285.0,142.5
4,kalshi,2026-05-14 22:40:00-04:00,40,285.0,142.5
5,kalshi,2026-05-14 22:45:00-04:00,40,285.0,142.5
6,kalshi,2026-05-14 22:50:00-04:00,40,285.0,142.5
7,kalshi,2026-05-14 22:55:00-04:00,38,285.0,142.5
8,kalshi,2026-05-14 23:00:00-04:00,40,285.0,142.5
9,kalshi,2026-05-14 23:05:00-04:00,40,285.0,142.5


In [8]:
book_validity(con)


,venue,snapshots,crossed_books,pct_crossed_books,invalid_price_rows,pct_invalid_price_rows,non_positive_spread_rows,pct_non_positive_spread,pct_empty_bid_levels,pct_empty_ask_levels,pct_with_top_both
0,kalshi,215126,0.0,0.0,0.0,0.0,0.0,0.0,0.066542,0.056023,0.877435
1,polymarket,4762335,0.0,0.0,0.0,0.0,0.0,0.0,0.189026,0.004276,0.807431


In [9]:
spread_depth_summary(con)


,venue,snapshots,spread_p50,spread_p90,spread_p99,spread_mean,avg_bid_depth,avg_ask_depth,avg_depth_imbalance
0,kalshi,215126,0.010,0.05,0.150,0.025169,2.846084e+04,38333.724437,-1.144395e+04
1,polymarket,4762335,0.002,0.02,0.088,0.009168,3.726516e+06,556893.507543,3.073517e+06


In [10]:
display(trade_overview(con))
metadata_universe(con)


,venue,trades,markets_with_trades,contracts,notional,first_trade_utc,latest_trade_utc,duplicate_trade_ids
0,polymarket,75368,509,4.510091e+07,7.159202e+06,2026-05-14 19:47:48.103788-04:00,2026-05-16 21:44:13.576000-04:00,1


,venue,status,metadata_rows,markets,avg_volume_24h,volume_24h_p50,volume_24h_p90,avg_liquidity,liquidity_p50,avg_hours_to_close
0,kalshi,active,4869,483,36472.803789,19937.420000,81279.28800,28973.171473,17913.28000,64.108159
1,kalshi,closed,2,2,0.000000,0.000000,0.00000,0.000000,0.00000,0.016667
2,polymarket,True,52002,1201,267624.394558,151077.987614,496173.55523,533362.879747,27574.15447,3010.546578


In [11]:
data_integrity_checks(con)


,table_name,check_name,issue_count
0,market_metadata_snapshots,null_critical_fields,0.0
1,order_book_snapshots,duplicate_snapshot_keys,0.0
2,order_book_snapshots,null_critical_fields,0.0
3,order_book_snapshots,schema_versions,1.0
4,trade_events,duplicate_trade_ids,53717.0
5,trade_events,null_critical_fields,0.0


In [12]:
readiness = analysis_readiness(
    con,
    min_snapshots=100,
    min_distinct_top_bids=2,
    max_spread_mean=0.10,
)
display(
    readiness.groupby(["venue", "is_analysis_ready", "exclusion_reason"])
    .size()
    .reset_index(name="markets")
    .sort_values(["venue", "is_analysis_ready", "markets"], ascending=[True, False, False])
)
readiness.head(20)


,venue,is_analysis_ready,exclusion_reason,markets
4,kalshi,True,ready,114
2,kalshi,False,no_top_book_movement,143
0,kalshi,False,incomplete_top_book,38
1,kalshi,False,low_snapshots,9
3,kalshi,False,wide_or_missing_spread,9
8,polymarket,True,ready,410
6,polymarket,False,no_top_book_movement,486
5,polymarket,False,incomplete_top_book,60
7,polymarket,False,wide_or_missing_spread,7


,venue,market_id,snapshots,distinct_top_bids,distinct_top_asks,spread_mean,pct_with_top_both,first_snapshot_utc,latest_snapshot_utc,is_analysis_ready,exclusion_reason
0,polymarket,0x0d6642ddd35287eb369945b9c2b00000b7526d341e3d...,11683,2,1,0.001001,1.0,2026-05-14 19:47:01.663487-04:00,2026-05-16 21:44:29.795213-04:00,True,ready
1,polymarket,0x0e4a0c937b8934c2475613b6322b3f8edc8dedc24762...,11683,4,4,0.010122,1.0,2026-05-14 19:47:01.663487-04:00,2026-05-16 21:44:29.795213-04:00,True,ready
2,polymarket,0x2785303a9349d892f464c251c1a39b838a8c535022c5...,11683,2,1,0.006178,1.0,2026-05-14 19:47:01.663487-04:00,2026-05-16 21:44:29.795213-04:00,True,ready
3,polymarket,0x3733a1b647e7364095736ab0966465d896a84cf3b6bc...,11683,51,52,0.010985,1.0,2026-05-14 19:47:01.663487-04:00,2026-05-16 21:44:29.795213-04:00,True,ready
4,polymarket,0x411d2f1251d86d9a4f30186c660745c597d18a19737f...,11683,5,6,0.069676,1.0,2026-05-14 19:47:01.663487-04:00,2026-05-16 21:44:29.795213-04:00,True,ready
5,polymarket,0x4e4a7df876b0c04f0b8b29b9073eddfbaf5c787192da...,11683,6,6,0.001000,1.0,2026-05-14 19:47:01.663487-04:00,2026-05-16 21:44:29.795213-04:00,True,ready
6,polymarket,0x518a5b030b205706b8ffe6bbad9bd3de59548348e5c0...,11683,21,21,0.007706,1.0,2026-05-14 19:47:01.663487-04:00,2026-05-16 21:44:29.795213-04:00,True,ready
7,polymarket,0x789c947a9415600d30d56a4aae88d4111996679b0cae...,11683,6,6,0.001006,1.0,2026-05-14 19:47:01.663487-04:00,2026-05-16 21:44:29.795213-04:00,True,ready
8,polymarket,0xafefe6fc14ffb9bd92f2ad0578696345ed3484a566d3...,11683,4,8,0.073903,1.0,2026-05-14 19:47:01.663487-04:00,2026-05-16 21:44:29.795213-04:00,True,ready
9,polymarket,0xbcacd5a055f5a9ced6f69f122216c073dd6987d08253...,11683,3,3,0.010009,1.0,2026-05-14 19:47:01.663487-04:00,2026-05-16 21:44:29.795213-04:00,True,ready


## Section 4: Market movement

In [13]:
movement = market_movement(con)
plot_movement_distribution(movement)

A market with no movement has repeated snapshots but no observed change in `top_bid` or `top_ask`. That can mean a stable liquid market, a stale market, or a capture path that is not receiving live updates. Inspect source breakdown and book depth before interpreting it as market behavior.

## Section 5: Pick a market and look at it

In [14]:
# Pick the market with the most top-of-book changes
most_active = movement.sort_values("distinct_top_bids", ascending=False).iloc[0]
market_id = most_active["market_id"]
print(
    f"Most active market: {market_id} (venue={most_active['venue']}, snapshots={most_active['snapshots']}, distinct_top_bids={most_active['distinct_top_bids']})"
)

aggs = multi_timescale_aggregation(con, market_id)
plot_multi_timescale_overlay(aggs)

Most active market: 0xbede2e0b0fd797304fe8f79fa12c0c2b4e19a2d440af0103c45224fdf4570845 (venue=polymarket, snapshots=1037, distinct_top_bids=183)


## Section 6: Live-watch active Kalshi Eurovision markets

This section watches the most actively-tracked Kalshi Eurovision ranking markets evolve over time. Eurovision 2026 resolves during the broadcast on Saturday evening (May 16). Run this cell repeatedly through the day — after each rsync run, the local archive will have more recent data and the chart will update accordingly.

To refresh the local archive manually before re-running this cell:

```bash
bash ~/ML/pm-edge/deploy/forward_indexer/rsync_to_laptop.sh
```

In [15]:
# Discover all Kalshi Eurovision markets currently in the captured data
eurovision_markets = con.sql("""
    SELECT market_id, COUNT(*) AS snapshots, MIN(timestamp_utc) AS first_seen
    FROM order_book_snapshots
    WHERE venue = 'kalshi' AND market_id LIKE 'KXEUROVISIONRANK%'
    GROUP BY market_id
    ORDER BY snapshots DESC
""").fetchdf()
eurovision_markets

,market_id,snapshots,first_seen
0,KXEUROVISIONRANK-26TOP10-MOL,5408,2026-05-15 19:40:49.129912-04:00
1,KXEUROVISIONRANK-26TOP10-AUS,5276,2026-05-15 19:40:49.129912-04:00
2,KXEUROVISIONRANK-26TOP10-ITA,4761,2026-05-15 19:40:49.129912-04:00
3,KXEUROVISIONRANK-26TOP5-AUS,4379,2026-05-15 19:40:49.129912-04:00
4,KXEUROVISIONRANK-26TOP5-ITA,3104,2026-05-15 19:40:49.129912-04:00
5,KXEUROVISIONRANK-26TOP10-BUL,1174,2026-05-16 13:54:19.968984-04:00
6,KXEUROVISIONRANK-26TOP10-ALB,915,2026-05-16 13:54:19.968984-04:00
7,KXEUROVISIONRANK-26TOP3-FRA,826,2026-05-16 15:34:34.721376-04:00
8,KXEUROVISIONRANK-26TOP3-MOL,760,2026-05-16 13:54:19.968984-04:00
9,KXEUROVISIONRANK-26TOP10-ISR,664,2026-05-16 15:34:34.721376-04:00


In [16]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if len(eurovision_markets) == 0:
    print("No Kalshi Eurovision markets found in the captured data. Nothing to plot.")
else:
    market_ids = eurovision_markets["market_id"].tolist()

    fig = make_subplots(
        rows=len(market_ids),
        cols=1,
        subplot_titles=market_ids,
        shared_xaxes=True,
        vertical_spacing=0.02,
    )

    for i, market_id in enumerate(market_ids, start=1):
        df = con.sql(f"""
            SELECT date_trunc('minute', timestamp_utc) AS minute,
                   AVG(top_bid) AS top_bid,
                   AVG(top_ask) AS top_ask
            FROM order_book_snapshots
            WHERE venue = 'kalshi' AND market_id = '{market_id}'
            GROUP BY minute
            ORDER BY minute
        """).fetchdf()

        fig.add_trace(
            go.Scatter(
                x=df["minute"],
                y=df["top_bid"],
                name="top_bid",
                line={"color": "blue"},
                showlegend=(i == 1),
            ),
            row=i,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=df["minute"],
                y=df["top_ask"],
                name="top_ask",
                line={"color": "red"},
                showlegend=(i == 1),
            ),
            row=i,
            col=1,
        )

    fig.update_layout(
        height=200 * len(market_ids),
        title="Eurovision rank markets — top_bid (blue) and top_ask (red) over time",
        margin={"t": 50, "b": 30, "l": 50, "r": 30},
    )
    fig.show()

## Section 6: Depth-dependent trade investigation

This section investigates trades whose size exceeds the contemporaneous level-1 opposing-side book size. The goal is to identify which markets, categories, and trade-size ranges drive the current depth-walking requirement for simulator v1.


### 6.1 What fraction of trades require depth, by venue and market?

Build a fresh-trade classification table using the latest prior book snapshot within 60 seconds, then rank markets by the fraction of trades larger than level-1 opposing-side size.


In [17]:
con.sql("""
    CREATE OR REPLACE TEMP TABLE depth_trade_classified AS
    WITH trades_with_book AS (
        SELECT
            t.venue,
            t.market_id,
            t.timestamp_utc,
            t.price,
            t.size AS trade_size,
            lower(t.side) AS side,
            b.bid_levels,
            b.ask_levels,
            EXTRACT(EPOCH FROM (t.timestamp_utc - b.timestamp_utc)) AS lag_seconds
        FROM trade_events t
        ASOF LEFT JOIN order_book_snapshots b
            ON t.market_id = b.market_id
            AND t.venue = b.venue
            AND t.timestamp_utc >= b.timestamp_utc
    )
    SELECT
        venue,
        market_id,
        timestamp_utc,
        price,
        trade_size,
        side,
        lag_seconds,
        CASE
            WHEN side = 'buy' AND len(ask_levels) > 0 THEN ask_levels[1].size
            WHEN side = 'sell' AND len(bid_levels) > 0 THEN bid_levels[1].size
            ELSE NULL
        END AS level_1_size,
        CASE
            WHEN side = 'buy' AND len(ask_levels) > 0 AND trade_size > ask_levels[1].size THEN 1
            WHEN side = 'sell' AND len(bid_levels) > 0 AND trade_size > bid_levels[1].size THEN 1
            ELSE 0
        END AS depth_required
    FROM trades_with_book
    WHERE lag_seconds <= 60
""")

depth_overall = con.sql("""
    SELECT
        venue,
        COUNT(*) AS total_trades,
        SUM(depth_required) AS depth_trades,
        SUM(depth_required)::DOUBLE / COUNT(*) AS depth_rate
    FROM depth_trade_classified
    WHERE level_1_size IS NOT NULL
    GROUP BY venue
    ORDER BY venue
""").fetchdf()

con.sql("""
    CREATE OR REPLACE TEMP TABLE depth_trades_view AS
    SELECT
        venue,
        market_id,
        COUNT(*) AS total_trades,
        SUM(depth_required) AS depth_trades,
        SUM(depth_required)::DOUBLE / COUNT(*) AS depth_rate,
        AVG(trade_size) AS avg_trade_size,
        AVG(level_1_size) AS avg_level_1_size
    FROM depth_trade_classified
    WHERE level_1_size IS NOT NULL
    GROUP BY venue, market_id
    HAVING COUNT(*) >= 20
""")

depth_trades = con.sql("""
    SELECT *
    FROM depth_trades_view
    ORDER BY depth_rate DESC, total_trades DESC
""").fetchdf()

display(depth_overall)
display(depth_trades.head(20))


,venue,total_trades,depth_trades,depth_rate
0,polymarket,70969,7307.0,0.10296


,venue,market_id,total_trades,depth_trades,depth_rate,avg_trade_size,avg_level_1_size
0,polymarket,0x8bb861046d70534248fe10aaef5ca28a91fe4c923db0...,184,112.0,0.608696,276.417105,170.390435
1,polymarket,0xc9615bb82d6535d630ff98e8094060bc767d8b944651...,415,245.0,0.590361,104.136946,146.189904
2,polymarket,0x707ab64115b0ca83b65bc5137aaa947a11f56885ed82...,152,88.0,0.578947,205.013616,138.328750
3,polymarket,0x60ecb823bb667a51ccf9ffcb1cab2c48899cd1641068...,65,33.0,0.507692,91.317824,194.295846
4,polymarket,0xff081c52486b2843da251a6d97187318a578af62f2a1...,250,117.0,0.468000,1410.955983,1127.174560
5,polymarket,0x718b2af2d2b6588ed650d191bf65fe598d70f0b87076...,32,13.0,0.406250,169.308954,248.871875
6,polymarket,0xeb91739b7afb5fdbdb72d0688f593e4ba3887f5e0ada...,68,26.0,0.382353,255.556268,303.128971
7,polymarket,0xb0041709dfdd0d8ae214fd1859b07c899ccd998522d8...,377,141.0,0.374005,635.279669,1000.294934
8,polymarket,0x79e7d16f5c71ab6ff5f805f77045eb5cef7b8f6aa87d...,41,15.0,0.365854,521.016243,982.169024
9,polymarket,0x2baed9d4160afbaf16209d5e2d309393904cd4815b1d...,233,84.0,0.360515,483.738847,899.241459


### 6.2 What does the depth-dependence rate look like as a distribution?

This histogram shows whether depth-dependence is concentrated in a small long tail of markets or spread broadly across the actively traded universe. The dashed reference line marks the full-archive v0 assumption-suite depth dependency rate of `10.32%`.


In [18]:
import plotly.express as px

if len(depth_trades) == 0:
    print("No markets met the minimum trade-count threshold for depth distribution.")
else:
    fig = px.histogram(
        depth_trades,
        x="depth_rate",
        color="venue",
        nbins=40,
        barmode="overlay",
        opacity=0.70,
        title="Distribution of depth-dependence rate by market",
        labels={"depth_rate": "Depth-dependent trade rate", "count": "Markets"},
    )
    fig.add_vline(
        x=0.1032,
        line_dash="dash",
        line_color="black",
        annotation_text="archive avg 10.32%",
    )
    fig.show()


### 6.3 How does depth-dependence correlate with trade size?

Bucket fresh trades by size and compute the depth-dependence rate in each bucket. This identifies where level-1-only simulation starts to understate fills materially.


In [19]:
size_buckets = con.sql("""
    SELECT
        venue,
        CASE
            WHEN trade_size < 10 THEN '00_under_10'
            WHEN trade_size < 100 THEN '01_10_to_100'
            WHEN trade_size < 1000 THEN '02_100_to_1000'
            WHEN trade_size < 10000 THEN '03_1000_to_10000'
            ELSE '04_over_10000'
        END AS size_bucket,
        COUNT(*) AS trades,
        SUM(depth_required) AS depth_trades,
        SUM(depth_required)::DOUBLE / COUNT(*) AS depth_rate
    FROM depth_trade_classified
    WHERE level_1_size IS NOT NULL
    GROUP BY venue, size_bucket
    ORDER BY venue, size_bucket
""").fetchdf()

display(size_buckets)

if len(size_buckets) == 0:
    print("No fresh trades with level-1 size available for size-bucket analysis.")
else:
    fig = px.bar(
        size_buckets,
        x="size_bucket",
        y="depth_rate",
        color="venue",
        barmode="group",
        text="trades",
        title="Depth-dependence rate by trade-size bucket",
        labels={
            "size_bucket": "Trade-size bucket",
            "depth_rate": "Depth-dependent trade rate",
            "trades": "Trades",
        },
    )
    fig.update_layout(xaxis_tickangle=-30)
    fig.show()


,venue,size_bucket,trades,depth_trades,depth_rate
0,polymarket,00_under_10,20543,259.0,0.012608
1,polymarket,01_10_to_100,28648,2320.0,0.080983
2,polymarket,02_100_to_1000,16315,3279.0,0.200981
3,polymarket,03_1000_to_10000,4776,1276.0,0.267169
4,polymarket,04_over_10000,687,173.0,0.251820


### 6.4 Can we identify the market categories where depth-dependence concentrates?

Polymarket market metadata does not expose a top-level category column, but `raw_json` includes structured Gamma fields. This section inspects those fields, then groups Polymarket by normalized `feeType` values such as `sports`, `politics`, `crypto`, `culture`, and `finance`. Kalshi tickers still use prefix parsing when trade data exists.


In [20]:
# Inspect available metadata fields and structured Polymarket category candidates
metadata_sample = con.sql("""
    SELECT
        venue,
        market_id,
        captured_at_utc,
        json_extract_string(raw_json, '$.feeType') AS fee_type,
        json_extract_string(raw_json, '$.question') AS question,
        json_extract_string(raw_json, '$.slug') AS slug,
        json_extract_string(raw_json, '$.groupItemTitle') AS group_item_title
    FROM market_metadata_snapshots
    WHERE venue = 'polymarket'
    LIMIT 5
""").fetchdf()

metadata_fee_types = con.sql("""
    SELECT
        json_extract_string(raw_json, '$.feeType') AS fee_type,
        COUNT(*) AS metadata_rows,
        COUNT(DISTINCT market_id) AS markets
    FROM market_metadata_snapshots
    WHERE venue = 'polymarket'
    GROUP BY fee_type
    ORDER BY markets DESC
""").fetchdf()

display(metadata_sample)
display(metadata_fee_types)


,venue,market_id,captured_at_utc,fee_type,question,slug,group_item_title
0,polymarket,0x0622b01925e56a091f248c42fb194d3b86d58165f664...,2026-05-14 19:45:50.155493-04:00,politics_fees,Will Mauricio Cardenas win the 2026 Colombian ...,will-mauricio-cardenas-win-the-2026-colombian-...,Mauricio Cárdenas
1,polymarket,0xa8599f4c633fb98f7543e04b20f91052564a6fbe8e02...,2026-05-14 19:45:50.155493-04:00,culture_fees,Will Richard Van De Water win The Bachelorette...,will-richard-van-de-water-win-the-bachelorette...,Richard Van De Water
2,polymarket,0x6a5589de888a72e9e1d711a323ceb1794c9135d4f4a2...,2026-05-14 19:45:50.155493-04:00,politics_fees,Will Al Mina be the Republican nominee for Sen...,will-al-mina-be-the-republican-nominee-for-sen...,Al Mina
3,polymarket,0x70f1a7d2cd490b640676d5e31470bcbe16fef40087de...,2026-05-14 19:45:50.155493-04:00,politics_fees,Will Eduardo Leite win the 2026 Brazilian pres...,will-eduardo-leite-win-the-2026-brazilian-pres...,Eduardo Leite
4,polymarket,0x1c842b91d226266fbe0e55d9bce9c52ce48b825d069e...,2026-05-14 19:45:50.155493-04:00,crypto_fees_v2,Metamask FDV above $700M one day after launch?,metamask-fdv-above-700m-one-day-after-launch-6...,$700M


,fee_type,metadata_rows,markets
0,sports_fees_v2,17256,389
1,politics_fees,12031,249
2,NaN,6855,163
3,culture_fees,4402,112
4,crypto_fees_v2,3947,110
5,finance_prices_fees,3364,80
6,tech_fees,2448,57
7,economics_fees,1304,34
8,weather_fees,313,6
9,general_fees,82,3


In [21]:
polymarket_category_sql = polymarket_category_expression("m.raw_json")
categorized = con.sql(f"""
    WITH latest_metadata AS (
        SELECT
            venue,
            market_id,
            raw_json,
            json_extract_string(raw_json, '$.feeType') AS fee_type,
            json_extract_string(raw_json, '$.question') AS question,
            json_extract_string(raw_json, '$.slug') AS slug,
            ROW_NUMBER() OVER (
                PARTITION BY venue, market_id
                ORDER BY captured_at_utc DESC
            ) AS rn
        FROM market_metadata_snapshots
    ),
    market_depth_with_category AS (
        SELECT
            d.venue,
            d.market_id,
            d.total_trades,
            d.depth_trades,
            d.depth_rate,
            CASE
                WHEN d.venue = 'kalshi' THEN regexp_extract(d.market_id, '^(KX[A-Z]+)', 1)
                WHEN d.venue = 'polymarket' THEN {polymarket_category_sql}
                ELSE 'uncategorized'
            END AS category,
            m.fee_type,
            m.question,
            m.slug
        FROM depth_trades_view d
        LEFT JOIN latest_metadata m
            ON d.venue = m.venue
            AND d.market_id = m.market_id
            AND m.rn = 1
    )
    SELECT
        venue,
        category,
        COUNT(*) AS markets,
        SUM(total_trades) AS total_trades,
        SUM(depth_trades) AS depth_trades,
        SUM(depth_trades)::DOUBLE / SUM(total_trades) AS weighted_depth_rate,
        AVG(depth_rate) AS avg_market_depth_rate
    FROM market_depth_with_category
    GROUP BY venue, category
    HAVING SUM(total_trades) >= 100
    ORDER BY weighted_depth_rate DESC, total_trades DESC
""").fetchdf()

categorized


,venue,category,markets,total_trades,depth_trades,weighted_depth_rate,avg_market_depth_rate
0,polymarket,geopolitics,27,5299.0,1184.0,0.223438,0.189734
1,polymarket,culture,45,9150.0,1490.0,0.162842,0.158510
2,polymarket,polymarket_uncategorized,7,3699.0,459.0,0.124088,0.147558
3,polymarket,politics,29,3522.0,431.0,0.122374,0.093619
4,polymarket,finance,8,2073.0,207.0,0.099855,0.090331
5,polymarket,sports,110,40696.0,3331.0,0.081851,0.072756
6,polymarket,crypto,10,1306.0,103.0,0.078867,0.111164
7,polymarket,economics,8,3313.0,13.0,0.003924,0.007122
8,polymarket,weather,1,796.0,2.0,0.002513,0.002513
9,polymarket,general,1,167.0,0.0,0.000000,0.000000


### Section 6 findings

_Fill this in after running the queries above._

Key observations:
- Overall depth-dependence rate: __%
- Top markets driving depth-dependence: __, __, __
- Distribution shape: [long tail / broad] — _Section 6.2_
- Size threshold for depth-walking importance: trades larger than __ contracts — _Section 6.3_
- Most affected categories: `polymarket_uncategorized`, `culture`, `politics`, `finance`, `sports` by weighted depth rate — _Section 6.4_

V1 simulator implication: [depth walking is essential for category X / depth walking is a long-tail edge case / etc.]


## Section 7: Uncategorized Polymarket markets — what are they?

Section 6.4 showed `polymarket_uncategorized` at 19.50% depth-dependence, higher than any classified category. This section investigates whether those markets share a coherent identity (e.g., a genuine market type we missed) or are a heterogeneous bucket (e.g., metadata-incomplete warm-up artifacts, retired markets, edge cases).

The answer determines whether the categorization logic should be extended or whether the uncategorized bucket is genuinely residual.


In [22]:
uncategorized_sample = con.sql("""
    WITH latest_metadata AS (
        SELECT
            venue,
            market_id,
            raw_json,
            captured_at_utc,
            ROW_NUMBER() OVER (
                PARTITION BY venue, market_id
                ORDER BY captured_at_utc DESC
            ) AS rn
        FROM market_metadata_snapshots
        WHERE venue = 'polymarket'
    )
    SELECT market_id, captured_at_utc, raw_json
    FROM latest_metadata
    WHERE rn = 1
      AND (
          json_extract_string(raw_json, '$.feeType') IS NULL
          OR json_extract_string(raw_json, '$.feeType') = ''
      )
    ORDER BY captured_at_utc DESC
    LIMIT 20
""").fetchdf()
uncategorized_sample


,market_id,captured_at_utc,raw_json
0,0xae6d228c3a89c04f5d48b130c86416b146ee3fdc19c6...,2026-05-16 21:37:28.852850-04:00,"{""acceptingOrders"": true, ""acceptingOrdersTime..."
1,0x30bd076067c87467e2437613f4ac9ce7b0ff529ffd1b...,2026-05-16 21:37:28.852850-04:00,"{""acceptingOrders"": true, ""acceptingOrdersTime..."
2,0x4ba348328e4d4ddee9e6734c9a369b2e8138611651f9...,2026-05-16 21:37:28.852850-04:00,"{""acceptingOrders"": true, ""acceptingOrdersTime..."
3,0xec4e5a2666b14fedded928dc4dcd6b5f2d38cc1eba91...,2026-05-16 21:37:28.852850-04:00,"{""acceptingOrders"": true, ""acceptingOrdersTime..."
4,0x85d949e1aa73caddf263dc356e6f93125f9889dc3af1...,2026-05-16 21:37:28.852850-04:00,"{""acceptingOrders"": true, ""acceptingOrdersTime..."
5,0xa1d97efb8a19de58d995edf58b882c4f99ef356c8a56...,2026-05-16 21:37:28.852850-04:00,"{""acceptingOrders"": true, ""acceptingOrdersTime..."
6,0x8b369e10358094a99ffe7f85a81a8e8ca68c611eee0f...,2026-05-16 21:37:28.852850-04:00,"{""acceptingOrders"": true, ""acceptingOrdersTime..."
7,0x90c9f679e2f272caf121b7f5ec4994525cbe58875314...,2026-05-16 21:37:28.852850-04:00,"{""acceptingOrders"": true, ""acceptingOrdersTime..."
8,0xfa8eeaa9872ac4fbe207d4ff0f559b3a59a3dc0a5350...,2026-05-16 21:37:28.852850-04:00,"{""acceptingOrders"": true, ""acceptingOrdersTime..."
9,0xd15f77a921a1c8f807ce2c53a5c24a60876ed383c429...,2026-05-16 21:37:28.852850-04:00,"{""acceptingOrders"": true, ""acceptingOrdersTime..."


In [23]:
import json

uncategorized_fields = {}
for _, row in uncategorized_sample.iterrows():
    try:
        parsed = json.loads(row["raw_json"])
        for key in parsed:
            uncategorized_fields[key] = uncategorized_fields.get(key, 0) + 1
    except (json.JSONDecodeError, TypeError):
        continue

field_freq = pd.DataFrame(
    [
        {"field": k, "count": v, "pct": v / len(uncategorized_sample) * 100}
        for k, v in uncategorized_fields.items()
    ]
).sort_values("count", ascending=False)
field_freq.head(30)


,field,count,pct
0,acceptingOrders,20,100.0
46,outcomePrices,20,100.0
52,ready,20,100.0
51,questionID,20,100.0
50,question,20,100.0
49,pendingDeployment,20,100.0
48,pagerDutyNotificationEnabled,20,100.0
47,outcomes,20,100.0
45,orderPriceMinTickSize,20,100.0
54,resolvedBy,20,100.0


In [24]:
samples = []
for _, row in uncategorized_sample.head(10).iterrows():
    try:
        parsed = json.loads(row["raw_json"])
        sample = {
            "market_id_short": row["market_id"][:20] + "...",
            "captured_at": row["captured_at_utc"],
        }
        for field in [
            "question",
            "title",
            "name",
            "description",
            "eventTitle",
            "event_title",
            "outcome",
        ]:
            if field in parsed:
                sample[field] = str(parsed[field])[:150]
                break
        samples.append(sample)
    except (json.JSONDecodeError, TypeError):
        continue

pd.DataFrame(samples)


,market_id_short,captured_at,question
0,0xae6d228c3a89c04f5d...,2026-05-16 21:37:28.852850-04:00,Will Muhammad Mirbaqiri be head of state in Ir...
1,0x30bd076067c87467e2...,2026-05-16 21:37:28.852850-04:00,Will Trump and Putin meet next in Australia?
2,0x4ba348328e4d4ddee9...,2026-05-16 21:37:28.852850-04:00,Iran closes its airspace by May 24?
3,0xec4e5a2666b14fedde...,2026-05-16 21:37:28.852850-04:00,Ukraine agrees to limit size of armed forces b...
4,0x85d949e1aa73caddf2...,2026-05-16 21:37:28.852850-04:00,Will Israel strike 6 countries in 2026?
5,0xa1d97efb8a19de58d9...,2026-05-16 21:37:28.852850-04:00,Kharg Island no longer under Iranian control b...
6,0x8b369e10358094a99f...,2026-05-16 21:37:28.852850-04:00,Will Donald Trump announce that the United Sta...
7,0x90c9f679e2f272caf1...,2026-05-16 21:37:28.852850-04:00,2k+ container ship transits of Suez Canal in H...
8,0xfa8eeaa9872ac4fbe2...,2026-05-16 21:37:28.852850-04:00,Will the next diplomatic US-Iran meeting be in...
9,0xd15f77a921a1c8f807...,2026-05-16 21:37:28.852850-04:00,Will there be at least 5000 measles cases in t...


In [25]:
uncategorized_depth_check = con.sql("""
    WITH latest_metadata AS (
        SELECT
            venue,
            market_id,
            raw_json,
            ROW_NUMBER() OVER (
                PARTITION BY venue, market_id
                ORDER BY captured_at_utc DESC
            ) AS rn
        FROM market_metadata_snapshots
        WHERE venue = 'polymarket'
    ),
    classified AS (
        SELECT
            d.market_id,
            d.total_trades,
            d.depth_rate,
            d.avg_trade_size,
            d.avg_level_1_size,
            CASE
                WHEN m.raw_json IS NULL THEN 'no_metadata_row'
                WHEN json_extract_string(m.raw_json, '$.feeType') IS NULL THEN 'no_feeType'
                WHEN json_extract_string(m.raw_json, '$.feeType') = '' THEN 'empty_feeType'
                ELSE 'has_feeType'
            END AS classification_status
        FROM depth_trades_view d
        LEFT JOIN latest_metadata m
            ON d.venue = m.venue
            AND d.market_id = m.market_id
            AND m.rn = 1
        WHERE d.venue = 'polymarket'
    )
    SELECT
        classification_status,
        COUNT(*) AS markets,
        SUM(total_trades) AS total_trades,
        AVG(depth_rate) AS avg_depth_rate,
        AVG(avg_trade_size) AS avg_trade_size,
        AVG(avg_level_1_size) AS avg_level_1_size
    FROM classified
    GROUP BY classification_status
    ORDER BY markets DESC
""").fetchdf()
uncategorized_depth_check


,classification_status,markets,total_trades,avg_depth_rate,avg_trade_size,avg_level_1_size
0,has_feeType,213,61066.0,0.092808,1679.416500,1.256309e+06
1,no_feeType,34,8998.0,0.181051,979.195631,2.195909e+04


### Section 7 findings

Run the queries above and fill in:

- **Uncategorized bucket composition:**
  - no_metadata_row: ___ markets (indexer coverage gap)
  - no_feeType: ___ markets (different market type)
  - empty_feeType: ___ markets (awaiting classification)

- **Coherence assessment:** [coherent — all share characteristic X / scattered — no clear pattern]
- **Recommended action:**
  - If coherent: extend categorization logic to capture them under a new label
  - If scattered: leave as residual, focus depth-walking work on identified high-depth categories instead
  - If indexer coverage gap: file TODO for indexer metadata capture improvement

- **Updated v1 priorities:** [depth-walking for culture markets remains priority / new category X needs attention / no change]


## Findings

_Fill this in after running the notebook._

### Eurovision watch notes

_Re-run Section 6 throughout the day on May 16 to observe how prediction-market top_bid and top_ask evolve as the Eurovision final approaches and resolves. Note: prices below 0.5 mean the market thinks the country is unlikely to make top 5/10; prices climbing toward 1.0 indicate growing confidence. Markets should converge to either 0 or 1 by resolution._